In [1]:
'''
Turn original .mat file into pymovements compatible .csv files.

Each .csv is a single trial.

Header definition:
timestamp  x  y  stimulus_index subject  choice  is_correct
'''

import os
import numpy as np
import pandas as pd
from scipy.io import loadmat

def convert_one(file_path, output_dir):

    SCREEN_WIDTH_PX  = 1920
    SCREEN_HEIGHT_PX = 1080
    SCREEN_WIDTH_CM  = 72.0
    SCREEN_HEIGHT_CM = 40.5

    PX_PER_CM_X = SCREEN_WIDTH_PX  / SCREEN_WIDTH_CM
    PX_PER_CM_Y = SCREEN_HEIGHT_PX / SCREEN_HEIGHT_CM
    
    print(f"Processing {os.path.split(file_path)[-1]}")
    mdata = loadmat(file_path, squeeze_me=True, struct_as_record=False)
    root_key = next(k for k in mdata if not k.startswith("__"))    # there will be only one key like "file009"
    content = mdata[root_key]
    raw_list = content.trialsData.eyeData_cm.nonBlinksData
    stimulus_index_list = content.trlInfo[1]
    choice_list = content.choice

    # Support for legacy data file processing, new data files use subject_id name, legacy files use filename
    try:
        subject_id = content.trialsParams[0].testSubjectInfo
        if isinstance(subject_id, str) and len(subject_id) > 0:
            subject_flag = 1
            filename = "none"
            subject_num = int(subject_id[1:])
            if subject_id.startswith('M'):
                subject_num += 100
            elif subject_id.startswith('F'):
                subject_num = subject_num
            else:
                raise(NameError("Subject id should start with 'M' or 'F'"))
        else:
            raise AttributeError
    except Exception:
        subject_flag = 0
        subject_num = 0
        filename = os.path.splitext(os.path.basename(file_path))[0]

    for idx, (trial_raw, sti_ind, choice) in enumerate(zip(raw_list, stimulus_index_list, choice_list)):

        # create dataframe for excel
        df = pd.DataFrame({
            "timestamp": np.arange(trial_raw.shape[0], dtype=int),
            "x": trial_raw[:, 0] * PX_PER_CM_X + SCREEN_WIDTH_PX // 2,
            "y": trial_raw[:, 1] * PX_PER_CM_Y + SCREEN_HEIGHT_PX // 2,
            "stimulus_index": sti_ind,
            "choice": int(choice),
            "subject_id": int(subject_num),
            "trial_id": int(idx),
            "filename": filename,
        })

        if subject_flag:
            output_file_path = os.path.join(output_dir, f"trial_{subject_num}_{idx}.csv")
            # write to csv
            df.to_csv(output_file_path, index=False)
        else:
            output_file_path = os.path.join(output_dir, f"trial_{filename}_{idx}.csv")
            df.to_csv(output_file_path, index=False)

if __name__ == '__main__':
    # raw .mat data path
    input_dir = r"Z:\BioMotionAnlyze\analyze\data\meta data\exp 202504\raw data\loadLvtRslt\trls"

    # output .csv data path(format can be processed by pymovements)
    base_dir = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504"

    # Create output directory if it doesn't exist
    output_dir = os.path.join(base_dir, "raw")
    os.makedirs(output_dir, exist_ok=True)

    print(os.listdir(input_dir))
    for file in os.listdir(input_dir):
        if file.endswith(".mat"):
            print(f"Processing {file}")
            file_path = os.path.join(input_dir, file)
            convert_one(file_path, output_dir)
    # convert_one(r"E:\BioMotion\data\processed0429\loadLvtRslt\trls\2025-04-28(122)-r0009-eyeTrcAnlyz.mat", output_dir)

['2025-04-28(122)-r0009-eyeTrcAnlyz.mat', '2025-04-28(122)-r0012-eyeTrcAnlyz.mat', '2025-04-28(122)-r0015-eyeTrcAnlyz.mat', '2025-04-28(122)-r0017-eyeTrcAnlyz.mat', '2025-04-28(122)-r0019-eyeTrcAnlyz.mat', '2025-04-29(122)-r0003-eyeTrcAnlyz.mat', '2025-04-29(122)-r0006-eyeTrcAnlyz.mat', '2025-04-29(122)-r0008-eyeTrcAnlyz.mat', '2025-04-29(122)-r0010-eyeTrcAnlyz.mat', '2025-04-29(122)-r0013-eyeTrcAnlyz.mat', '2025-04-29(122)-r0018-eyeTrcAnlyz.mat', '2025-04-29(122)-r0021-eyeTrcAnlyz.mat', '2025-04-29(122)-r0026-eyeTrcAnlyz.mat', '2025-04-30(122)-r0008-eyeTrcAnlyz.mat', '2025-04-30(122)-r0011-eyeTrcAnlyz.mat', '2025-04-30(122)-r0014-eyeTrcAnlyz.mat', '2025-11-24(122)-r0004-eyeTrcAnlyz.mat', '2025-11-24(122)-r0006-eyeTrcAnlyz.mat', '2025-11-24(122)-r0010-eyeTrcAnlyz.mat', '2025-11-25(122)-r0007-eyeTrcAnlyz.mat', '2025-11-25(122)-r0020-eyeTrcAnlyz.mat', '2025-11-26(122)-r0030-eyeTrcAnlyz.mat', '2025-11-28(122)-r0003-eyeTrcAnlyz.mat']
Processing 2025-04-28(122)-r0009-eyeTrcAnlyz.mat
Process

In [4]:
# 修正性别信息相同的覆盖问题

'''
Turn original .mat file into pymovements compatible .csv files.

Each .csv is a single trial.

Header definition:
timestamp  x  y  stimulus_index subject  choice  is_correct
'''

import os
import numpy as np
import pandas as pd
from scipy.io import loadmat

# ============================================================
# OUTPUT NAMING MODE (ONLY FEATURE ADDED)
# ============================================================
# "auto":
#   - If NO valid gender code (old data) -> use source filename stem as core
#   - If HAS valid gender code (new data) -> follow NAMING_FOR_CODED
#
# "filename":
#   - Always use source filename stem as core
#
# "coded":
#   - Always use gender-based numeric subject id as core (e.g., 115 / 11)
OUTPUT_NAMING_MODE = "filename"   # "auto" | "filename" | "coded"

# Only used when OUTPUT_NAMING_MODE == "auto" and gender code exists
NAMING_FOR_CODED = "coded"    # "filename" | "coded"


# ============================================================
# SUBJECT / NAMING HELPERS
# ============================================================
def get_subject_info_from_content(content, file_path):
    """
    Try to read subject code from content.trialsParams[0].testSubjectInfo.

    Valid codes:
      - 'Mxx' -> numeric subject_num = xx + 100
      - 'Fxx' -> numeric subject_num = xx

    Fallback:
      - If invalid or missing -> subject_num = 0, has_gender_code = False

    Returns:
      subject_num (int), has_gender_code (bool), filename_stem (str)
    """
    filename_stem = os.path.splitext(os.path.basename(file_path))[0]

    try:
        subject_id = content.trialsParams[0].testSubjectInfo
        if isinstance(subject_id, str):
            s = subject_id.strip()
        else:
            s = ""

        if len(s) >= 2 and (s[0] in ("M", "F")) and s[1:].isdigit():
            num = int(s[1:])
            if s.startswith("M"):
                num += 100
            # F -> keep as is
            return num, True, filename_stem

        # invalid format -> treat as no gender code
        return 0, False, filename_stem

    except Exception:
        return 0, False, filename_stem


def choose_output_core(subject_num, has_gender_code, filename_stem):
    """
    Decide the output core used in CSV filenames: trial_<core>_<trial_idx>.csv

    core candidates:
      - coded core:    str(subject_num)   (e.g., "115")
      - filename core: filename_stem      (e.g., "2025-04-28(122)-r0009-eyeTrcAnlyz")
    """
    coded_core = str(int(subject_num))  # ensure numeric string
    filename_core = filename_stem

    if OUTPUT_NAMING_MODE == "filename":
        return filename_core

    if OUTPUT_NAMING_MODE == "coded":
        return coded_core

    # auto mode
    if not has_gender_code:
        # old/legacy data -> force filename-based core
        return filename_core

    # new data -> follow user preference
    return coded_core if NAMING_FOR_CODED == "coded" else filename_core


# ============================================================
# MAIN CONVERTER
# ============================================================
def convert_one(file_path, output_dir):

    SCREEN_WIDTH_PX  = 1920
    SCREEN_HEIGHT_PX = 1080
    SCREEN_WIDTH_CM  = 72.0
    SCREEN_HEIGHT_CM = 40.5

    PX_PER_CM_X = SCREEN_WIDTH_PX  / SCREEN_WIDTH_CM
    PX_PER_CM_Y = SCREEN_HEIGHT_PX / SCREEN_HEIGHT_CM

    print(f"Processing {os.path.split(file_path)[-1]}")
    mdata = loadmat(file_path, squeeze_me=True, struct_as_record=False)
    root_key = next(k for k in mdata if not k.startswith("__"))    # only one key like "file009"
    content = mdata[root_key]

    raw_list = content.trialsData.eyeData_cm.nonBlinksData
    stimulus_index_list = content.trlInfo[1]
    choice_list = content.choice

    # --- naming decision (ONLY change) ---
    subject_num, has_gender_code, filename_stem = get_subject_info_from_content(content, file_path)
    core = choose_output_core(subject_num, has_gender_code, filename_stem)

    # Keep these fields for downstream usage/debugging (unchanged behavior)
    filename_for_df = "none" if has_gender_code else filename_stem

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    for idx, (trial_raw, sti_ind, choice) in enumerate(zip(raw_list, stimulus_index_list, choice_list)):

        df = pd.DataFrame({
            "timestamp": np.arange(trial_raw.shape[0], dtype=int),
            "x": trial_raw[:, 0] * PX_PER_CM_X + SCREEN_WIDTH_PX // 2,
            "y": trial_raw[:, 1] * PX_PER_CM_Y + SCREEN_HEIGHT_PX // 2,
            "stimulus_index": sti_ind,
            "choice": int(choice),
            "subject_id": int(subject_num),     # note: for legacy data this remains 0 (same idea as before)
            "trial_id": int(idx),
            "filename": filename_for_df,        # "none" if coded, else filename stem
        })

        # Strict naming: no extra suffix/version/timestamp
        output_file_path = os.path.join(output_dir, f"trial_{core}_{idx}.csv")
        df.to_csv(output_file_path, index=False)


if __name__ == '__main__':
    # raw .mat data path
    input_dir = r"Z:\BioMotionAnlyze\analyze\data\meta data\202511_typeA\loadLvtRslt\trls\rt"

    # output .csv data path(format can be processed by pymovements)
    base_dir = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\rt"

    # Create output directory if it doesn't exist
    output_dir = os.path.join(base_dir, "raw")
    os.makedirs(output_dir, exist_ok=True)

    for file in os.listdir(input_dir):
        if file.endswith(".mat"):
            file_path = os.path.join(input_dir, file)
            convert_one(file_path, output_dir)

    # convert_one(r"E:\BioMotion\data\processed0429\loadLvtRslt\trls\2025-04-28(122)-r0009-eyeTrcAnlyz.mat", output_dir)


Processing 2025-11-25(122)-r0025-eyeTrcAnlyz.mat
Processing 2025-11-25(122)-r0029-eyeTrcAnlyz.mat
Processing 2025-11-25(122)-r0032-eyeTrcAnlyz.mat
Processing 2025-11-25(122)-r0037-eyeTrcAnlyz.mat
Processing 2025-11-25(122)-r0040-eyeTrcAnlyz.mat
Processing 2025-11-25(122)-r0043-eyeTrcAnlyz.mat
Processing 2025-11-25(122)-r0048-eyeTrcAnlyz.mat
Processing 2025-11-25(122)-r0053-eyeTrcAnlyz.mat
Processing 2025-11-25(122)-r0056-eyeTrcAnlyz.mat
Processing 2025-11-26(122)-r0004-eyeTrcAnlyz.mat
Processing 2025-11-26(122)-r0007-eyeTrcAnlyz.mat
Processing 2025-11-26(122)-r0021-eyeTrcAnlyz.mat
Processing 2025-11-26(122)-r0024-eyeTrcAnlyz.mat
Processing 2025-11-26(122)-r0027-eyeTrcAnlyz.mat
Processing 2025-12-05(122)-r0004-eyeTrcAnlyz.mat
Processing 2025-12-05(122)-r0007-eyeTrcAnlyz.mat
Processing 2025-12-05(122)-r0011-eyeTrcAnlyz.mat
Processing 2025-12-05(122)-r0014-eyeTrcAnlyz.mat
Processing 2025-12-05(122)-r0018-eyeTrcAnlyz.mat
Processing 2025-12-05(122)-r0021-eyeTrcAnlyz.mat
Processing 2025-12-0